In [1]:
from calendar import day_name
from webbrowser import open_new

import pandas as pd
import numpy as np
from datetime import timedelta


In [2]:
customers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_items_dataset.csv')
order_payment = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_payments_dataset.csv')
order_review = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_sellers_dataset.csv')
products_category_name_translation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/product_category_name_translation.csv')


In [3]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [4]:
order_review.sample()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
4183,c4651c892d941ed7a422649798f51744,b0b55c59d28b64d3b611cb034eb9f1e6,3,NaN,NaN,2017-09-13 00:00:00,2017-09-15 16:56:30


In [5]:
order_review['review_score'].value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [6]:
all_df = order_review.merge(orders,on='order_id', how='right').merge(order_items,on='order_id', how='right').merge(products,on='product_id',how='right').merge(products_category_name_translation,on='product_category_name',how='right')
all_df.sample()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_id,order_status,order_purchase_timestamp,...,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
28812,41b80ae9d42a291d78de4a0ce8403f5a,51122623ee1ccdaadbb6fcd95f077d8a,5.0,NaN,Serviço rápido e eficiente.,2018-06-26 00:00:00,2018-06-26 22:01:20,bfa4d52a5bd197fec9654daa897d31ff,delivered,2018-06-20 10:08:53,...,41.33,cama_mesa_banho,53.0,597.0,1.0,6550.0,30.0,30.0,30.0,bed_bath_table


In [7]:
review_category = (
    order_items[['order_id', 'product_id']]
    .merge(
        products[['product_id', 'product_category_name']],
        on='product_id',
        how='left'
    )
    .merge(
        products_category_name_translation,
        on='product_category_name',
        how='left'
    )
    [['order_id', 'product_category_name_english']]
    .dropna(subset=['product_category_name_english'])
    .drop_duplicates()
)

review_category = review_category.merge(
    order_review[['order_id', 'review_score']],
    on='order_id',
    how='inner'
)

review_category.groupby(
    'product_category_name_english'
)['review_score'].mean().sort_values().head(10)

product_category_name_english
security_and_services                2.500000
office_furniture                     3.617508
fashion_male_clothing                3.702703
fashio_female_clothing               3.731707
diapers_and_hygiene                  3.740741
furniture_mattress_and_upholstery    3.815789
home_comfort_2                       3.826087
audio                                3.827586
construction_tools_safety            3.849398
home_confort                         3.859296
Name: review_score, dtype: float64

In [8]:
orders.sample()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
35146,2ce81d16edb6568f5ff5b9ec2e5cb4b7,a2a5fc7e8fc7e880cd68a9768b8caa22,delivered,2017-12-28 14:45:48,2017-12-28 14:55:33,2018-01-03 23:23:01,2018-01-08 16:28:42,2018-01-19


In [9]:
all_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 111678 entries, 0 to 111677
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   review_id                      110750 non-null  str           
 1   order_id                       111678 non-null  str           
 2   review_score                   110750 non-null  float64       
 3   review_comment_title           13347 non-null   str           
 4   review_comment_message         46923 non-null   str           
 5   review_creation_date           110750 non-null  str           
 6   review_answer_timestamp        110750 non-null  str           
 7   customer_id                    111678 non-null  str           
 8   order_status                   111678 non-null  str           
 9   order_purchase_timestamp       111678 non-null  datetime64[us]
 10  order_approved_at              111664 non-null  str           
 11  order_deliv

In [10]:
all_df['delivery_days'] = (
    all_df['order_delivered_customer_date']
    - all_df['order_purchase_timestamp']
).dt.total_seconds() / 86400

In [11]:
new_alldf = all_df[['order_id','product_category_name','product_category_name_english','review_score','delivery_days']]
new_alldf.sample(10)
new_alldf.groupby('product_category_name_english').agg({'review_score':'mean','delivery_days':'mean'}).sort_values(by='review_score')

,review_score,delivery_days
product_category_name_english,,
security_and_services,2.500000,15.138154
diapers_and_hygiene,3.256410,10.775192
office_furniture,3.493183,20.854464
home_comfort_2,3.629630,14.573525
fashion_male_clothing,3.641221,12.951343
...,...,...
flowers,4.419355,11.147610
costruction_tools_tools,4.444444,12.075415
books_general_interest,4.446266,11.613084


In [12]:
orders['delivery_status'] = np.where(orders['order_delivered_customer_date']<= orders['order_estimated_delivery_date'],'On Time','Late')
review_delivery = order_review.merge(orders[['order_id', 'delivery_status']],on='order_id',how='inner')
review_delivery.groupby('delivery_status')['review_score'].mean().sort_values()

delivery_status
Late       2.348192
On Time    4.293578
Name: review_score, dtype: float64

In [13]:
review_status = order_review.merge(orders[['order_id', 'order_status']],on='order_id',how='inner')
review_status.groupby('order_status')['review_score'].agg(['mean', 'count']).sort_values(by='mean')

,mean,count
order_status,,
processing,1.277027,296
unavailable,1.530988,597
invoiced,1.661342,313
canceled,1.811166,609
shipped,2.008629,1043
created,2.333333,3
approved,2.500000,2
delivered,4.155717,96361
